In [2]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import two_tower_training
importlib.reload(two_tower_training)

from two_tower_training import (
    TrainConfig,
    compare_on_common_objectives,
    export_frozen_embeddings,
    load_processed_two_tower_data,
    run_config_grid,
    summarize_results,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PREFIX = "vehicle_sensor_subset_200ms"
RUN_DIR = PROJECT_ROOT / "experiments" / "runs" / "two_tower"

# Set to a small integer like 800 for quick iteration; use None for the full run.
MAX_TIME_STEPS = None

data = load_processed_two_tower_data(
    PROCESSED_DIR,
    prefix=PREFIX,
    utility_name="saved",
    max_time_steps=MAX_TIME_STEPS,
)

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("context_dim:", data["C_by_time"].shape[1])
print("action_dim:", data["A_examples"].shape[1])
print("times:", len(data["C_by_time"]))
print("examples:", len(data["A_examples"]))
print("actions/time:", data["meta"]["num_actions_per_time"])


device: cuda
context_dim: 118
action_dim: 69
times: 7465
examples: 306065
actions/time: 41


In [5]:
# Compact sweep. Each epoch prints one diagnostics line per active run.
# Utility options supported by the training module:
#   saved, rational, clipped_linear, closest_binary, rank_discount

CONFIGS = [
    TrainConfig(
        run_name="saved_h64_e8",
        utility_name="saved",
        hidden=64,
        emb_dim=8,
        depth=2,
        dropout=0.05,
        combine_mode="mul_only",
        loss_name="mse",
        max_epochs=25,
        patience=8,
        batch_size=2048,
        lr=1e-3,
        weight_decay=1e-4,
        log_every=1,
    ),
    TrainConfig(
        run_name="clip_h64_e8",
        utility_name="clipped_linear",
        utility_kwargs={"utility_radius": 30.0},
        hidden=64,
        emb_dim=8,
        depth=2,
        dropout=0.05,
        combine_mode="mul_only",
        loss_name="mse",
        max_epochs=25,
        patience=8,
        batch_size=2048,
        lr=1e-3,
        weight_decay=1e-4,
        log_every=1,
    ),
    TrainConfig(
        run_name="binary_h64_e8",
        utility_name="closest_binary",
        hidden=64,
        emb_dim=8,
        depth=2,
        dropout=0.05,
        combine_mode="mul_only",
        loss_name="bce",
        max_epochs=25,
        patience=8,
        batch_size=2048,
        lr=1e-3,
        weight_decay=1e-4,
        log_every=1,
    ),
]

results = run_config_grid(
    PROCESSED_DIR,
    CONFIGS,
    prefix=PREFIX,
    max_time_steps=MAX_TIME_STEPS,
)

own_target_summary_df = summarize_results(results)
print("Own-target diagnostics only; use the common-objective table below to compare utilities.")
display(own_target_summary_df)


            saved_h64_e8 ep 001/025* loss=0.03131 val_rmse=0.1479 top1=0.186 top3=0.422 reg=0.0403 rank=6.04 9s
            saved_h64_e8 ep 002/025* loss=0.01700 val_rmse=0.1443 top1=0.172 top3=0.404 reg=0.0370 rank=5.81 17s
            saved_h64_e8 ep 003/025* loss=0.01355 val_rmse=0.1423 top1=0.227 top3=0.438 reg=0.0348 rank=5.65 26s
            saved_h64_e8 ep 004/025  loss=0.01120 val_rmse=0.1404 top1=0.246 top3=0.427 reg=0.0352 rank=5.79 36s
            saved_h64_e8 ep 005/025* loss=0.00962 val_rmse=0.1404 top1=0.268 top3=0.472 reg=0.0318 rank=5.39 45s
            saved_h64_e8 ep 006/025  loss=0.00834 val_rmse=0.1413 top1=0.269 top3=0.455 reg=0.0341 rank=5.72 54s
            saved_h64_e8 ep 007/025* loss=0.00747 val_rmse=0.1381 top1=0.280 top3=0.472 reg=0.0316 rank=5.47 64s
            saved_h64_e8 ep 008/025* loss=0.00677 val_rmse=0.1404 top1=0.302 top3=0.499 reg=0.0302 rank=5.28 73s
            saved_h64_e8 ep 009/025  loss=0.00627 val_rmse=0.1424 top1=0.305 top3=0.488 reg=0.030

,run_name,utility,hidden,emb_dim,depth,dropout,combine,loss,best_epoch,train_rmse,...,val_avg_regret,val_avg_norm_regret,test_rmse,test_mae,test_r2,test_top1,test_top3,test_mean_rank,test_avg_regret,test_avg_norm_regret
0,clip_h64_e8,clipped_linear,64,8,2,0.05,mul_only,mse,8,0.097192,...,0.020967,0.045176,0.230155,0.154219,0.357555,0.750837,0.760884,3.485599,0.048644,0.094660
1,saved_h64_e8,saved,64,8,2,0.05,mul_only,mse,14,0.054120,...,0.027523,0.059756,0.138637,0.096998,0.514722,0.318151,0.496316,5.026792,0.032868,0.061710
2,binary_h64_e8,closest_binary,64,8,2,0.05,mul_only,bce,3,0.244501,...,0.073677,0.073677,0.405288,0.231275,0.309705,0.884796,0.884796,2.843269,0.115204,0.115204


In [6]:
# Compare every trained run on the same downstream objectives.
# top1: the model-selected subset is truly best for that objective.
# top3: the model-selected subset is within the true top 3 actions.
# mean_rank: the true-utility rank of the model-selected subset.

COMMON_OBJECTIVES = [
    {"name": "contains_closest", "utility_name": "closest_binary", "utility_kwargs": {}},
    {"name": "rank_discount", "utility_name": "rank_discount", "utility_kwargs": {}},
    {"name": "clipped_r30", "utility_name": "clipped_linear", "utility_kwargs": {"utility_radius": 30.0}},
    {"name": "saved_rational", "utility_name": "saved", "utility_kwargs": {}},
]

common_eval_df = compare_on_common_objectives(results, COMMON_OBJECTIVES)
display(common_eval_df)

test_policy_df = (
    common_eval_df[common_eval_df["split"].eq("test")]
    .sort_values(
        ["eval_objective", "top1", "top3", "mean_rank", "avg_norm_regret"],
        ascending=[True, False, False, True, True],
    )
    .reset_index(drop=True)
)
display(test_policy_df)

PRIMARY_OBJECTIVE = "saved_rational"
PRIMARY_SPLIT = "val"
primary_candidates = common_eval_df[
    common_eval_df["eval_objective"].eq(PRIMARY_OBJECTIVE)
    & common_eval_df["split"].eq(PRIMARY_SPLIT)
].copy()

best_row = primary_candidates.sort_values(
    ["top1", "top3", "mean_rank", "avg_norm_regret"],
    ascending=[False, False, True, True],
).iloc[0]
best_name = best_row["run_name"]
best_result = next(result for result in results if result["config"].run_name == best_name)

print("primary objective:", PRIMARY_OBJECTIVE)
print("best run:", best_name)
print("best epoch:", best_result["best_epoch"])
print("validation policy metrics:", best_row[["top1", "top3", "mean_rank", "avg_regret", "avg_norm_regret"]].to_dict())

display(
    common_eval_df[
        common_eval_df["run_name"].eq(best_name)
        & common_eval_df["split"].eq("test")
    ].sort_values("eval_objective")
)
display(best_result["history"].tail(8))


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,saved_h64_e8,saved,clipped_r30,test,64,8,mul_only,0.793704,0.802411,3.004689,0.039383,0.076319
1,clip_h64_e8,clipped_linear,clipped_r30,test,64,8,mul_only,0.750837,0.760884,3.485599,0.048644,0.094660
2,binary_h64_e8,closest_binary,clipped_r30,test,64,8,mul_only,0.724715,0.735432,3.830543,0.062842,0.111279
3,clip_h64_e8,clipped_linear,clipped_r30,val,64,8,mul_only,0.819156,0.828533,2.427997,0.020967,0.045176
4,saved_h64_e8,saved,clipped_r30,val,64,8,mul_only,0.808439,0.817147,2.515070,0.022956,0.051010
5,binary_h64_e8,closest_binary,clipped_r30,val,64,8,mul_only,0.771601,0.779638,3.123242,0.036830,0.074280
6,saved_h64_e8,saved,contains_closest,test,64,8,mul_only,0.920965,0.920965,2.264568,0.079035,0.079035
7,clip_h64_e8,clipped_linear,contains_closest,test,64,8,mul_only,0.886135,0.886135,2.821835,0.113865,0.113865
8,binary_h64_e8,closest_binary,contains_closest,test,64,8,mul_only,0.884796,0.884796,2.843269,0.115204,0.115204
9,clip_h64_e8,clipped_linear,contains_closest,val,64,8,mul_only,0.940388,0.940388,1.953784,0.059612,0.059612


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,saved_h64_e8,saved,clipped_r30,test,64,8,mul_only,0.793704,0.802411,3.004689,0.039383,0.076319
1,clip_h64_e8,clipped_linear,clipped_r30,test,64,8,mul_only,0.750837,0.760884,3.485599,0.048644,0.094660
2,binary_h64_e8,closest_binary,clipped_r30,test,64,8,mul_only,0.724715,0.735432,3.830543,0.062842,0.111279
3,saved_h64_e8,saved,contains_closest,test,64,8,mul_only,0.920965,0.920965,2.264568,0.079035,0.079035
4,clip_h64_e8,clipped_linear,contains_closest,test,64,8,mul_only,0.886135,0.886135,2.821835,0.113865,0.113865
5,binary_h64_e8,closest_binary,contains_closest,test,64,8,mul_only,0.884796,0.884796,2.843269,0.115204,0.115204
6,saved_h64_e8,saved,rank_discount,test,64,8,mul_only,0.920965,0.920965,2.519089,0.043313,0.051976
7,clip_h64_e8,clipped_linear,rank_discount,test,64,8,mul_only,0.886135,0.886135,3.222371,0.062715,0.075258
8,binary_h64_e8,closest_binary,rank_discount,test,64,8,mul_only,0.884796,0.884796,3.359678,0.065171,0.078205
9,saved_h64_e8,saved,saved_rational,test,64,8,mul_only,0.318151,0.496316,5.026792,0.032868,0.061710


primary objective: contains_closest
best run: clip_h64_e8
best epoch: 8
validation policy metrics: {'top1': 0.9403884795713329, 'top3': 0.9403884795713329, 'mean_rank': 1.9537843268586739, 'avg_regret': 0.05961152042866711, 'avg_norm_regret': 0.05961152042866711}


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
1,clip_h64_e8,clipped_linear,clipped_r30,test,64,8,mul_only,0.750837,0.760884,3.485599,0.048644,0.094660
7,clip_h64_e8,clipped_linear,contains_closest,test,64,8,mul_only,0.886135,0.886135,2.821835,0.113865,0.113865
13,clip_h64_e8,clipped_linear,rank_discount,test,64,8,mul_only,0.886135,0.886135,3.222371,0.062715,0.075258
19,clip_h64_e8,clipped_linear,saved_rational,test,64,8,mul_only,0.115874,0.300067,7.737441,0.055021,0.109720


,epoch,train_loss,val_rmse,val_mae,val_r2,val_top1,val_top3,val_mean_rank,val_avg_regret,val_avg_norm_regret
8,9,0.011936,0.216345,0.140074,0.393883,0.788346,0.797723,2.737441,0.025847,0.055426
9,10,0.011055,0.216693,0.140147,0.391932,0.793704,0.802411,2.642331,0.024857,0.053172
10,11,0.010327,0.213980,0.138108,0.407059,0.805090,0.815137,2.573342,0.023774,0.050440
11,12,0.009830,0.213730,0.136703,0.408447,0.804421,0.813798,2.643001,0.024942,0.053741
12,13,0.009310,0.216306,0.139520,0.394101,0.787676,0.796383,2.839920,0.028314,0.062765
13,14,0.008746,0.213109,0.137055,0.411880,0.814468,0.824514,2.558607,0.023246,0.053458
14,15,0.008262,0.212849,0.137305,0.413313,0.814468,0.825184,2.633624,0.023977,0.055050
15,16,0.007783,0.217599,0.139071,0.386835,0.804421,0.817147,2.634963,0.026312,0.056491


In [8]:
# Saved-utility width sweep. Depth stays at the TrainConfig default: depth=2.

SAVED_OBJECTIVES = [
    {"name": "saved_rational", "utility_name": "saved", "utility_kwargs": {}},
    {"name": "contains_closest", "utility_name": "closest_binary", "utility_kwargs": {}},
]

HIDDEN_SWEEP_CONFIGS = [
    TrainConfig(
        run_name=f"saved_h{hidden}_d2",
        utility_name="saved",
        hidden=hidden,
        emb_dim=8,
        depth=2,
        dropout=0.05,
        combine_mode="mul_only",
        loss_name="mse",
        max_epochs=50,
        patience=16,
        batch_size=2048,
        lr=1e-3,
        weight_decay=1e-4,
        log_every=1,
    )
    for hidden in [128, 256, 512, 1024]
]

hidden_sweep_results = run_config_grid(
    PROCESSED_DIR,
    HIDDEN_SWEEP_CONFIGS,
    prefix=PREFIX,
    max_time_steps=MAX_TIME_STEPS,
)

hidden_sweep_own_df = summarize_results(hidden_sweep_results)
hidden_sweep_eval_df = compare_on_common_objectives(hidden_sweep_results, SAVED_OBJECTIVES)
hidden_sweep_test_df = (
    hidden_sweep_eval_df[hidden_sweep_eval_df["split"].eq("test")]
    .sort_values(
        ["eval_objective", "top1", "top3", "mean_rank", "avg_norm_regret"],
        ascending=[True, False, False, True, True],
    )
    .reset_index(drop=True)
)

display(hidden_sweep_own_df)
display(hidden_sweep_test_df)


           saved_h128_d2 ep 001/050* loss=0.02948 val_rmse=0.1530 top1=0.143 top3=0.296 reg=0.0484 rank=7.14 10s
           saved_h128_d2 ep 002/050* loss=0.01542 val_rmse=0.1530 top1=0.176 top3=0.354 reg=0.0435 rank=6.51 19s
           saved_h128_d2 ep 003/050* loss=0.01186 val_rmse=0.1470 top1=0.211 top3=0.431 reg=0.0391 rank=6.09 28s
           saved_h128_d2 ep 004/050  loss=0.00960 val_rmse=0.1460 top1=0.203 top3=0.404 reg=0.0412 rank=6.41 37s
           saved_h128_d2 ep 005/050  loss=0.00800 val_rmse=0.1426 top1=0.213 top3=0.417 reg=0.0406 rank=6.39 46s
           saved_h128_d2 ep 006/050  loss=0.00683 val_rmse=0.1421 top1=0.232 top3=0.461 reg=0.0397 rank=6.21 55s
           saved_h128_d2 ep 007/050  loss=0.00587 val_rmse=0.1449 top1=0.258 top3=0.480 reg=0.0396 rank=6.10 65s
           saved_h128_d2 ep 008/050  loss=0.00517 val_rmse=0.1463 top1=0.283 top3=0.520 reg=0.0397 rank=5.97 75s
           saved_h128_d2 ep 009/050  loss=0.00464 val_rmse=0.1444 top1=0.251 top3=0.479 reg=0.03

,run_name,utility,hidden,emb_dim,depth,dropout,combine,loss,best_epoch,train_rmse,...,val_avg_regret,val_avg_norm_regret,test_rmse,test_mae,test_r2,test_top1,test_top3,test_mean_rank,test_avg_regret,test_avg_norm_regret
0,saved_h1024_d2,saved,1024,8,2,0.05,mul_only,mse,19,0.018852,...,0.026392,0.056875,0.136741,0.097498,0.527903,0.346283,0.550569,5.058272,0.037365,0.068837
1,saved_h128_d2,saved,128,8,2,0.05,mul_only,mse,23,0.030477,...,0.031929,0.067503,0.145943,0.104194,0.462229,0.249833,0.472204,5.634963,0.038814,0.075557
2,saved_h512_d2,saved,512,8,2,0.05,mul_only,mse,5,0.058980,...,0.032708,0.069746,0.136276,0.099437,0.531108,0.213664,0.423309,5.402545,0.033447,0.065294
3,saved_h256_d2,saved,256,8,2,0.05,mul_only,mse,3,0.081391,...,0.036389,0.076080,0.141017,0.102714,0.497920,0.172806,0.381782,5.914936,0.037923,0.071975


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,saved_h512_d2,saved,contains_closest,test,512,8,mul_only,0.919625,0.919625,2.286001,0.080375,0.080375
1,saved_h256_d2,saved,contains_closest,test,256,8,mul_only,0.914267,0.914267,2.371735,0.085733,0.085733
2,saved_h128_d2,saved,contains_closest,test,128,8,mul_only,0.899531,0.899531,2.607502,0.100469,0.100469
3,saved_h1024_d2,saved,contains_closest,test,1024,8,mul_only,0.890824,0.890824,2.746818,0.109176,0.109176
4,saved_h1024_d2,saved,saved_rational,test,1024,8,mul_only,0.346283,0.550569,5.058272,0.037365,0.068837
5,saved_h128_d2,saved,saved_rational,test,128,8,mul_only,0.249833,0.472204,5.634963,0.038814,0.075557
6,saved_h512_d2,saved,saved_rational,test,512,8,mul_only,0.213664,0.423309,5.402545,0.033447,0.065294
7,saved_h256_d2,saved,saved_rational,test,256,8,mul_only,0.172806,0.381782,5.914936,0.037923,0.071975


In [9]:
print(next(results[0]["model"].parameters()).device)
print(torch.cuda.memory_allocated() / 1e9)
print(torch.cuda.memory_reserved() / 1e9)

cuda:0
0.02486272
0.121634816


In [10]:
# Saved-utility depth sweep. Hidden width stays at the TrainConfig default: hidden=64.

DEPTH_SWEEP_CONFIGS = [
    TrainConfig(
        run_name=f"saved_h64_d{depth}",
        utility_name="saved",
        hidden=64,
        emb_dim=8,
        depth=depth,
        dropout=0.05,
        combine_mode="mul_only",
        loss_name="mse",
        max_epochs=50,
        patience=16,
        batch_size=16384,
        lr=1e-3,
        weight_decay=1e-4,
        log_every=1,
    )
    for depth in [3, 4, 5, 6]
]

depth_sweep_results = run_config_grid(
    PROCESSED_DIR,
    DEPTH_SWEEP_CONFIGS,
    prefix=PREFIX,
    max_time_steps=MAX_TIME_STEPS,
)

depth_sweep_own_df = summarize_results(depth_sweep_results)
depth_sweep_eval_df = compare_on_common_objectives(depth_sweep_results, SAVED_OBJECTIVES)
depth_sweep_test_df = (
    depth_sweep_eval_df[depth_sweep_eval_df["split"].eq("test")]
    .sort_values(
        ["eval_objective", "top1", "top3", "mean_rank", "avg_norm_regret"],
        ascending=[True, False, False, True, True],
    )
    .reset_index(drop=True)
)

display(depth_sweep_own_df)
display(depth_sweep_test_df)


            saved_h64_d3 ep 001/050* loss=0.07259 val_rmse=0.1725 top1=0.064 top3=0.206 reg=0.0947 rank=10.37 8s
            saved_h64_d3 ep 002/050* loss=0.03431 val_rmse=0.1665 top1=0.117 top3=0.254 reg=0.0776 rank=8.99 15s
            saved_h64_d3 ep 003/050* loss=0.02903 val_rmse=0.1617 top1=0.174 top3=0.332 reg=0.0616 rank=7.70 22s
            saved_h64_d3 ep 004/050* loss=0.02599 val_rmse=0.1564 top1=0.194 top3=0.382 reg=0.0531 rank=7.11 29s
            saved_h64_d3 ep 005/050* loss=0.02373 val_rmse=0.1543 top1=0.175 top3=0.380 reg=0.0501 rank=7.00 36s
            saved_h64_d3 ep 006/050  loss=0.02187 val_rmse=0.1545 top1=0.150 top3=0.376 reg=0.0508 rank=7.15 44s
            saved_h64_d3 ep 007/050  loss=0.02030 val_rmse=0.1519 top1=0.154 top3=0.391 reg=0.0505 rank=7.04 51s
            saved_h64_d3 ep 008/050* loss=0.01887 val_rmse=0.1509 top1=0.159 top3=0.396 reg=0.0476 rank=6.93 58s
            saved_h64_d3 ep 009/050* loss=0.01770 val_rmse=0.1501 top1=0.151 top3=0.393 reg=0.04

,run_name,utility,hidden,emb_dim,depth,dropout,combine,loss,best_epoch,train_rmse,...,val_avg_regret,val_avg_norm_regret,test_rmse,test_mae,test_r2,test_top1,test_top3,test_mean_rank,test_avg_regret,test_avg_norm_regret
0,saved_h64_d4,saved,64,8,4,0.05,mul_only,mse,49,0.055641,...,0.032966,0.070627,0.143397,0.101274,0.480826,0.271266,0.449431,5.716678,0.040112,0.076856
1,saved_h64_d3,saved,64,8,3,0.05,mul_only,mse,49,0.056627,...,0.035475,0.076201,0.143941,0.103108,0.476883,0.237776,0.421299,5.673141,0.036669,0.072332
2,saved_h64_d5,saved,64,8,5,0.05,mul_only,mse,44,0.057085,...,0.040149,0.082697,0.148402,0.105566,0.443949,0.248493,0.430676,5.724046,0.038005,0.073585
3,saved_h64_d6,saved,64,8,6,0.05,mul_only,mse,9,0.130751,...,0.041902,0.084057,0.149039,0.114374,0.439170,0.168788,0.381782,6.995981,0.052599,0.099164


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,saved_h64_d5,saved,contains_closest,test,64,8,mul_only,0.905559,0.905559,2.511052,0.094441,0.094441
1,saved_h64_d3,saved,contains_closest,test,64,8,mul_only,0.902210,0.902210,2.564635,0.097790,0.097790
2,saved_h64_d4,saved,contains_closest,test,64,8,mul_only,0.896182,0.896182,2.661085,0.103818,0.103818
3,saved_h64_d6,saved,contains_closest,test,64,8,mul_only,0.874079,0.874079,3.014735,0.125921,0.125921
4,saved_h64_d4,saved,saved_rational,test,64,8,mul_only,0.271266,0.449431,5.716678,0.040112,0.076856
5,saved_h64_d5,saved,saved_rational,test,64,8,mul_only,0.248493,0.430676,5.724046,0.038005,0.073585
6,saved_h64_d3,saved,saved_rational,test,64,8,mul_only,0.237776,0.421299,5.673141,0.036669,0.072332
7,saved_h64_d6,saved,saved_rational,test,64,8,mul_only,0.168788,0.381782,6.995981,0.052599,0.099164


In [5]:
# Saved-utility embedding sweep. Hidden/depth stay at TrainConfig defaults: hidden=64, depth=2.
DEVICE = 'cuda'

SAVED_OBJECTIVES = [
    {"name": "saved_rational", "utility_name": "saved", "utility_kwargs": {}},
    {"name": "contains_closest", "utility_name": "closest_binary", "utility_kwargs": {}},
]

EMBEDDING_SWEEP_CONFIGS = [
    TrainConfig(
        run_name=f"saved_h64_d2_e{emb_dim}",
        utility_name="saved",
        hidden=64,
        emb_dim=emb_dim,
        depth=2,
        dropout=0.05,
        combine_mode="mul_only",
        loss_name="mse",
        max_epochs=50,
        patience=16,
        batch_size=8192,
        lr=1e-3,
        weight_decay=1e-4,
        log_every=1,
    )
    for emb_dim in [16, 32, 64, 96]
]

embedding_sweep_results = run_config_grid(
    PROCESSED_DIR,
    EMBEDDING_SWEEP_CONFIGS,
    prefix=PREFIX,
    max_time_steps=MAX_TIME_STEPS,
    device=DEVICE,
)

embedding_sweep_own_df = summarize_results(embedding_sweep_results)
embedding_sweep_eval_df = compare_on_common_objectives(embedding_sweep_results, SAVED_OBJECTIVES)
embedding_sweep_test_df = (
    embedding_sweep_eval_df[embedding_sweep_eval_df["split"].eq("test")]
    .sort_values(
        ["eval_objective", "top1", "top3", "mean_rank", "avg_norm_regret"],
        ascending=[True, False, False, True, True],
    )
    .reset_index(drop=True)
)

display(embedding_sweep_own_df)
display(embedding_sweep_test_df)


        saved_h64_d2_e16 ep 001/050* loss=0.12516 val_rmse=0.1802 top1=0.065 top3=0.224 reg=0.1260 rank=12.21 8s
        saved_h64_d2_e16 ep 002/050* loss=0.03387 val_rmse=0.1655 top1=0.078 top3=0.381 reg=0.0684 rank=8.32 16s
        saved_h64_d2_e16 ep 003/050* loss=0.02722 val_rmse=0.1551 top1=0.096 top3=0.374 reg=0.0497 rank=7.04 24s
        saved_h64_d2_e16 ep 004/050* loss=0.02411 val_rmse=0.1533 top1=0.100 top3=0.368 reg=0.0451 rank=6.69 32s
        saved_h64_d2_e16 ep 005/050* loss=0.02191 val_rmse=0.1502 top1=0.095 top3=0.346 reg=0.0429 rank=6.59 41s
        saved_h64_d2_e16 ep 006/050* loss=0.02026 val_rmse=0.1486 top1=0.105 top3=0.338 reg=0.0422 rank=6.56 49s
        saved_h64_d2_e16 ep 007/050* loss=0.01891 val_rmse=0.1497 top1=0.113 top3=0.330 reg=0.0419 rank=6.50 58s
        saved_h64_d2_e16 ep 008/050* loss=0.01776 val_rmse=0.1466 top1=0.124 top3=0.338 reg=0.0409 rank=6.36 66s
        saved_h64_d2_e16 ep 009/050* loss=0.01678 val_rmse=0.1467 top1=0.130 top3=0.342 reg=0.04

NameError: name 'SAVED_OBJECTIVES' is not defined

In [6]:
SAVED_OBJECTIVES = [
    {"name": "saved_rational", "utility_name": "saved", "utility_kwargs": {}},
    {"name": "contains_closest", "utility_name": "closest_binary", "utility_kwargs": {}},
]

embedding_sweep_own_df = summarize_results(embedding_sweep_results)
embedding_sweep_eval_df = compare_on_common_objectives(
    embedding_sweep_results,
    SAVED_OBJECTIVES,
)

embedding_sweep_test_df = (
    embedding_sweep_eval_df[embedding_sweep_eval_df["split"].eq("test")]
    .sort_values(
        ["eval_objective", "top1", "top3", "mean_rank", "avg_norm_regret"],
        ascending=[True, False, False, True, True],
    )
    .reset_index(drop=True)
)

display(embedding_sweep_own_df)
display(embedding_sweep_test_df)

,run_name,utility,hidden,emb_dim,depth,dropout,combine,loss,best_epoch,train_rmse,...,val_avg_regret,val_avg_norm_regret,test_rmse,test_mae,test_r2,test_top1,test_top3,test_mean_rank,test_avg_regret,test_avg_norm_regret
0,saved_h64_d2_e16,saved,64,16,2,0.05,mul_only,mse,41,0.064033,...,0.030820,0.065023,0.144651,0.104246,0.471710,0.251172,0.445412,5.501674,0.036026,0.071060
1,saved_h64_d2_e32,saved,64,32,2,0.05,mul_only,mse,12,0.084477,...,0.031475,0.065655,0.141825,0.103295,0.492150,0.207636,0.418620,5.637642,0.036882,0.069524
2,saved_h64_d2_e64,saved,64,64,2,0.05,mul_only,mse,25,0.060218,...,0.034535,0.073889,0.142028,0.100700,0.490691,0.131279,0.344943,6.019424,0.042712,0.079012
3,saved_h64_d2_e96,saved,64,96,2,0.05,mul_only,mse,12,0.087122,...,0.036180,0.075328,0.142927,0.104996,0.484226,0.149364,0.356999,6.079035,0.036553,0.073278


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,saved_h64_d2_e32,saved,contains_closest,test,64,32,mul_only,0.924313,0.924313,2.210985,0.075687,0.075687
1,saved_h64_d2_e96,saved,contains_closest,test,64,96,mul_only,0.918955,0.918955,2.296718,0.081045,0.081045
2,saved_h64_d2_e64,saved,contains_closest,test,64,64,mul_only,0.905559,0.905559,2.511052,0.094441,0.094441
3,saved_h64_d2_e16,saved,contains_closest,test,64,16,mul_only,0.902210,0.902210,2.564635,0.097790,0.097790
4,saved_h64_d2_e16,saved,saved_rational,test,64,16,mul_only,0.251172,0.445412,5.501674,0.036026,0.071060
5,saved_h64_d2_e32,saved,saved_rational,test,64,32,mul_only,0.207636,0.418620,5.637642,0.036882,0.069524
6,saved_h64_d2_e96,saved,saved_rational,test,64,96,mul_only,0.149364,0.356999,6.079035,0.036553,0.073278
7,saved_h64_d2_e64,saved,saved_rational,test,64,64,mul_only,0.131279,0.344943,6.019424,0.042712,0.079012


In [8]:
# Saved-utility stability sweep. Compare the simple baseline against the strongest width candidate.
# This keeps the utility fixed and checks whether architecture differences survive seed variation.

SAVED_OBJECTIVES = [
    {"name": "saved_rational", "utility_name": "saved", "utility_kwargs": {}},
    {"name": "contains_closest", "utility_name": "closest_binary", "utility_kwargs": {}},
]

SEEDS = [11, 22, 33, 44, 55]
SEED_SWEEP_CONFIGS = []

for seed in SEEDS:
    SEED_SWEEP_CONFIGS.append(
        TrainConfig(
            run_name=f"saved_h64_d2_e8_seed{seed}",
            utility_name="saved",
            hidden=64,
            emb_dim=8,
            depth=2,
            dropout=0.05,
            combine_mode="mul_only",
            loss_name="mse",
            max_epochs=50,
            patience=16,
            batch_size=8192,
            lr=1e-3,
            weight_decay=1e-4,
            seed=seed,
            log_every=1,
        )
    )
    SEED_SWEEP_CONFIGS.append(
        TrainConfig(
            run_name=f"saved_h512_d2_e8_seed{seed}",
            utility_name="saved",
            hidden=512,
            emb_dim=8,
            depth=2,
            dropout=0.05,
            combine_mode="mul_only",
            loss_name="mse",
            max_epochs=50,
            patience=16,
            batch_size=8192,
            lr=1e-3,
            weight_decay=1e-4,
            seed=seed,
            log_every=1,
        )
    )

seed_sweep_results = run_config_grid(
    PROCESSED_DIR,
    SEED_SWEEP_CONFIGS,
    prefix=PREFIX,
    max_time_steps=MAX_TIME_STEPS,
    device=DEVICE,
)

seed_sweep_own_df = summarize_results(seed_sweep_results)
seed_sweep_eval_df = compare_on_common_objectives(seed_sweep_results, SAVED_OBJECTIVES)

seed_sweep_test_df = (
    seed_sweep_eval_df[seed_sweep_eval_df["split"].eq("test")]
    .sort_values(
        ["eval_objective", "top1", "top3", "mean_rank", "avg_norm_regret"],
        ascending=[True, False, False, True, True],
    )
    .reset_index(drop=True)
)

seed_sweep_summary_df = (
    seed_sweep_eval_df[seed_sweep_eval_df["split"].eq("test")]
    .assign(architecture=lambda df: df["hidden"].astype(str).radd("h") + "_e" + df["emb_dim"].astype(str))
    .groupby(["eval_objective", "architecture"], as_index=False)
    .agg(
        top1_mean=("top1", "mean"),
        top1_std=("top1", "std"),
        top3_mean=("top3", "mean"),
        top3_std=("top3", "std"),
        mean_rank_mean=("mean_rank", "mean"),
        mean_rank_std=("mean_rank", "std"),
        avg_regret_mean=("avg_regret", "mean"),
        avg_regret_std=("avg_regret", "std"),
        avg_norm_regret_mean=("avg_norm_regret", "mean"),
        avg_norm_regret_std=("avg_norm_regret", "std"),
    )
    .sort_values(["eval_objective", "avg_norm_regret_mean", "mean_rank_mean"])
    .reset_index(drop=True)
)

display(seed_sweep_own_df)
display(seed_sweep_test_df)
display(seed_sweep_summary_df)


  saved_h64_d2_e8_seed11 ep 001/050* loss=0.07894 val_rmse=0.1678 top1=0.116 top3=0.242 reg=0.1037 rank=10.60 10s
  saved_h64_d2_e8_seed11 ep 002/050* loss=0.03131 val_rmse=0.1575 top1=0.205 top3=0.374 reg=0.0681 rank=8.09 19s
  saved_h64_d2_e8_seed11 ep 003/050* loss=0.02538 val_rmse=0.1505 top1=0.190 top3=0.384 reg=0.0549 rank=7.36 28s
  saved_h64_d2_e8_seed11 ep 004/050* loss=0.02235 val_rmse=0.1489 top1=0.186 top3=0.395 reg=0.0505 rank=7.14 37s
  saved_h64_d2_e8_seed11 ep 005/050* loss=0.02031 val_rmse=0.1465 top1=0.190 top3=0.407 reg=0.0467 rank=6.81 45s
  saved_h64_d2_e8_seed11 ep 006/050* loss=0.01878 val_rmse=0.1455 top1=0.196 top3=0.417 reg=0.0451 rank=6.70 53s
  saved_h64_d2_e8_seed11 ep 007/050* loss=0.01738 val_rmse=0.1445 top1=0.182 top3=0.417 reg=0.0440 rank=6.63 61s
  saved_h64_d2_e8_seed11 ep 008/050* loss=0.01619 val_rmse=0.1440 top1=0.173 top3=0.416 reg=0.0426 rank=6.52 69s
  saved_h64_d2_e8_seed11 ep 009/050* loss=0.01521 val_rmse=0.1429 top1=0.175 top3=0.409 reg=0.0

,run_name,utility,hidden,emb_dim,depth,dropout,combine,loss,best_epoch,train_rmse,...,val_avg_regret,val_avg_norm_regret,test_rmse,test_mae,test_r2,test_top1,test_top3,test_mean_rank,test_avg_regret,test_avg_norm_regret
0,saved_h512_d2_e8_seed11,saved,512,8,2,0.05,mul_only,mse,35,0.033868,...,0.027523,0.060771,0.140221,0.100297,0.503570,0.262559,0.470194,5.296718,0.037829,0.070760
1,saved_h64_d2_e8_seed22,saved,64,8,2,0.05,mul_only,mse,28,0.065148,...,0.028372,0.061452,0.137434,0.098280,0.523109,0.308104,0.460817,5.182184,0.032348,0.061981
2,saved_h64_d2_e8_seed11,saved,64,8,2,0.05,mul_only,mse,35,0.065579,...,0.031021,0.064959,0.140620,0.101201,0.500741,0.183523,0.406564,5.776959,0.038139,0.074261
3,saved_h512_d2_e8_seed44,saved,512,8,2,0.05,mul_only,mse,7,0.081699,...,0.031890,0.066005,0.142707,0.105567,0.485813,0.158071,0.398526,5.570663,0.033920,0.066559
4,saved_h64_d2_e8_seed44,saved,64,8,2,0.05,mul_only,mse,49,0.054820,...,0.031993,0.069547,0.139696,0.099810,0.507278,0.248493,0.462827,5.746149,0.040587,0.078086
5,saved_h512_d2_e8_seed22,saved,512,8,2,0.05,mul_only,mse,27,0.041826,...,0.032077,0.066986,0.138532,0.099658,0.515453,0.237776,0.447421,5.308774,0.033974,0.065375
6,saved_h512_d2_e8_seed55,saved,512,8,2,0.05,mul_only,mse,7,0.080270,...,0.032193,0.067174,0.138647,0.102510,0.514653,0.155392,0.385131,5.773610,0.036228,0.070815
7,saved_h64_d2_e8_seed33,saved,64,8,2,0.05,mul_only,mse,47,0.059334,...,0.032260,0.068749,0.138139,0.099659,0.518204,0.292699,0.498995,5.163429,0.031643,0.062075
8,saved_h512_d2_e8_seed33,saved,512,8,2,0.05,mul_only,mse,18,0.053185,...,0.033694,0.069860,0.138144,0.100665,0.518166,0.207636,0.444742,5.352311,0.034415,0.067061
9,saved_h64_d2_e8_seed55,saved,64,8,2,0.05,mul_only,mse,31,0.069340,...,0.034894,0.074090,0.141327,0.104547,0.495705,0.183523,0.421969,5.820496,0.037913,0.073786


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,saved_h512_d2_e8_seed44,saved,contains_closest,test,512,8,mul_only,0.929002,0.929002,2.135968,0.070998,0.070998
1,saved_h64_d2_e8_seed22,saved,contains_closest,test,64,8,mul_only,0.926993,0.926993,2.168118,0.073007,0.073007
2,saved_h64_d2_e8_seed33,saved,contains_closest,test,64,8,mul_only,0.922304,0.922304,2.243135,0.077696,0.077696
3,saved_h512_d2_e8_seed22,saved,contains_closest,test,512,8,mul_only,0.918955,0.918955,2.296718,0.081045,0.081045
4,saved_h512_d2_e8_seed55,saved,contains_closest,test,512,8,mul_only,0.917616,0.917616,2.318151,0.082384,0.082384
5,saved_h512_d2_e8_seed33,saved,contains_closest,test,512,8,mul_only,0.916276,0.916276,2.339585,0.083724,0.083724
6,saved_h64_d2_e8_seed55,saved,contains_closest,test,64,8,mul_only,0.908908,0.908908,2.457468,0.091092,0.091092
7,saved_h64_d2_e8_seed11,saved,contains_closest,test,64,8,mul_only,0.904889,0.904889,2.521768,0.095111,0.095111
8,saved_h512_d2_e8_seed11,saved,contains_closest,test,512,8,mul_only,0.901541,0.901541,2.575352,0.098459,0.098459
9,saved_h64_d2_e8_seed44,saved,contains_closest,test,64,8,mul_only,0.886135,0.886135,2.821835,0.113865,0.113865


,eval_objective,architecture,top1_mean,top1_std,top3_mean,top3_std,mean_rank_mean,mean_rank_std,avg_regret_mean,avg_regret_std,avg_norm_regret_mean,avg_norm_regret_std
0,contains_closest,h512_e8,0.916678,0.009839,0.916678,0.009839,2.333155,0.157429,0.083322,0.009839,0.083322,0.009839
1,contains_closest,h64_e8,0.909846,0.016100,0.909846,0.016100,2.442465,0.257602,0.090154,0.016100,0.090154,0.016100
2,saved_rational,h512_e8,0.204287,0.047579,0.429203,0.035837,5.460415,0.207198,0.035273,0.001710,0.068114,0.002516
3,saved_rational,h64_e8,0.243269,0.058765,0.450234,0.036586,5.537843,0.334342,0.036126,0.003922,0.070038,0.007500


In [9]:
# Pick the best saved-utility architecture from whichever sweeps have been run.

SAVED_OBJECTIVES = [
    {"name": "saved_rational", "utility_name": "saved", "utility_kwargs": {}},
    {"name": "contains_closest", "utility_name": "closest_binary", "utility_kwargs": {}},
]

saved_arch_results = []
for result_var in [
    "hidden_sweep_results",
    "depth_sweep_results",
    "embedding_sweep_results",
    "seed_sweep_results",
]:
    saved_arch_results.extend(globals().get(result_var, []))

if not saved_arch_results:
    raise RuntimeError("Run at least one saved-utility sweep before picking a best architecture.")

saved_arch_eval_df = compare_on_common_objectives(saved_arch_results, SAVED_OBJECTIVES)

saved_arch_val_df = saved_arch_eval_df[
    saved_arch_eval_df["split"].eq("val")
    & saved_arch_eval_df["eval_objective"].eq("saved_rational")
].copy()

best_arch_row = saved_arch_val_df.sort_values(
    ["avg_norm_regret", "mean_rank", "top3", "top1"],
    ascending=[True, True, False, False],
).iloc[0]
best_saved_arch_name = best_arch_row["run_name"]
best_saved_arch_result = next(
    result for result in saved_arch_results
    if result["config"].run_name == best_saved_arch_name
)

print("best saved-utility architecture:", best_saved_arch_name)
print("validation saved metrics:", best_arch_row[["top1", "top3", "mean_rank", "avg_regret", "avg_norm_regret"]].to_dict())

display(
    saved_arch_eval_df[saved_arch_eval_df["split"].eq("test")]
    .sort_values(["eval_objective", "avg_norm_regret", "mean_rank"])
    .reset_index(drop=True)
)


best saved-utility architecture: saved_h512_d2_e8_seed11
validation saved metrics: {'top1': 0.27059611520428667, 'top3': 0.5097119892833222, 'mean_rank': 4.983255190890824, 'avg_regret': 0.027523223328175198, 'avg_norm_regret': 0.060770917688785696}


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,saved_h512_d2_e8_seed44,saved,contains_closest,test,512,8,mul_only,0.929002,0.929002,2.135968,0.070998,0.070998
1,saved_h64_d2_e8_seed22,saved,contains_closest,test,64,8,mul_only,0.926993,0.926993,2.168118,0.073007,0.073007
2,saved_h64_d2_e32,saved,contains_closest,test,64,32,mul_only,0.924313,0.924313,2.210985,0.075687,0.075687
3,saved_h64_d2_e8_seed33,saved,contains_closest,test,64,8,mul_only,0.922304,0.922304,2.243135,0.077696,0.077696
4,saved_h64_d2_e96,saved,contains_closest,test,64,96,mul_only,0.918955,0.918955,2.296718,0.081045,0.081045
5,saved_h512_d2_e8_seed22,saved,contains_closest,test,512,8,mul_only,0.918955,0.918955,2.296718,0.081045,0.081045
6,saved_h512_d2_e8_seed55,saved,contains_closest,test,512,8,mul_only,0.917616,0.917616,2.318151,0.082384,0.082384
7,saved_h512_d2_e8_seed33,saved,contains_closest,test,512,8,mul_only,0.916276,0.916276,2.339585,0.083724,0.083724
8,saved_h64_d2_e8_seed55,saved,contains_closest,test,64,8,mul_only,0.908908,0.908908,2.457468,0.091092,0.091092
9,saved_h64_d2_e64,saved,contains_closest,test,64,64,mul_only,0.905559,0.905559,2.511052,0.094441,0.094441


In [ ]:
# Save the trained towers, standardizers, and frozen embeddings for fast scoring.
# The embeddings file includes context embeddings by time and action embeddings by example.

export_result = globals().get("best_saved_arch_result", best_result)

paths = export_frozen_embeddings(
    export_result,
    output_dir=RUN_DIR,
    prefix=export_result["config"].run_name,
)

manifest = pd.DataFrame(
    [{"artifact": key, "path": str(path)} for key, path in paths.items()]
)
display(manifest)
